In [6]:
!git clone https://github.com/fajarowaes/adaptive-sensor-fusion.git

Cloning into 'adaptive-sensor-fusion'...


In [7]:
%cd adaptive-sensor-fusion

/content/adaptive-sensor-fusion


In [9]:
import pandas as pd
import numpy as np
import requests
import zipfile
import re
from io import BytesIO
from pathlib import Path

# --------------------------------------------------
# 1. Download the MotionSense dataset
# --------------------------------------------------

data_url = "https://github.com/mmalekzadeh/motion-sense/raw/master/data/A_DeviceMotion_data.zip"

subject_url = "https://raw.githubusercontent.com/mmalekzadeh/motion-sense/master/data/data_subjects_info.csv"

response = requests.get(data_url, timeout=120)
response.raise_for_status()

zip_file = zipfile.ZipFile(BytesIO(response.content))

print("Dataset downloaded successfully.")

# --------------------------------------------------
# 2. Define activity labels and sensor columns
# --------------------------------------------------

activity_labels = {
    "dws": "downstairs",
    "ups": "upstairs",
    "wlk": "walking",
    "jog": "jogging",
    "std": "standing",
    "sit": "sitting"
}

sensor_columns = [
    "attitude.roll",
    "attitude.pitch",
    "attitude.yaw",
    "gravity.x",
    "gravity.y",
    "gravity.z",
    "rotationRate.x",
    "rotationRate.y",
    "rotationRate.z",
    "userAcceleration.x",
    "userAcceleration.y",
    "userAcceleration.z"
]

# --------------------------------------------------
# 3. Read and combine valid sensor CSV files
# --------------------------------------------------

all_data = []
skipped_files = 0

for file in zip_file.namelist():

    # Only process CSV files inside the sensor data folder
    if not file.endswith(".csv"):
        continue

    if "A_DeviceMotion_data/" not in file:
        continue

    file_name = Path(file).name

    # Skip macOS metadata files such as ._13.csv
    if file_name.startswith("._") or "__MACOSX" in file:
        skipped_files += 1
        continue

    # Accept only filenames such as sub_13.csv
    match = re.fullmatch(r"sub_(\d+)\.csv", file_name)

    if match is None:
        skipped_files += 1
        continue

    subject_id = int(match.group(1))
    folder_name = Path(file).parent.name

    # Extract activity and trial number
    folder_parts = folder_name.split("_")

    if len(folder_parts) < 2:
        skipped_files += 1
        continue

    activity_code = folder_parts[0]

    if activity_code not in activity_labels:
        skipped_files += 1
        continue

    try:
        trial = int(folder_parts[-1])
    except ValueError:
        skipped_files += 1
        continue

    # Read CSV file
    df = pd.read_csv(zip_file.open(file))

    # Rename unnecessary index column
    if "Unnamed: 0" in df.columns:
        df = df.rename(
            columns={"Unnamed: 0": "sample_index"}
        )

    # Check whether required sensor columns exist
    if not all(col in df.columns for col in sensor_columns):
        print("Skipping file with missing columns:", file)
        skipped_files += 1
        continue

    # Add activity and participant information
    df["activity"] = activity_labels[activity_code]
    df["activity_code"] = activity_code
    df["subject_id"] = subject_id
    df["trial"] = trial
    df["source_file"] = file

    all_data.append(df)

# Combine all valid files
if len(all_data) == 0:
    raise ValueError(
        "No valid sensor CSV files found. "
        "Check the ZIP file structure."
    )

data = pd.concat(all_data, ignore_index=True)

print("Valid files loaded:", len(all_data))
print("Files skipped:", skipped_files)
print("Combined dataset shape:", data.shape)

# --------------------------------------------------
# 4. Standardize column names
# --------------------------------------------------

data.columns = data.columns.str.strip()

# --------------------------------------------------
# 5. Convert sensor columns to numeric
# --------------------------------------------------

for col in sensor_columns:
    data[col] = pd.to_numeric(
        data[col], errors="coerce"
    )

# Replace infinite values with NaN
data[sensor_columns] = data[sensor_columns].replace(
    [np.inf, -np.inf], np.nan
)

print("\nMissing sensor values before cleaning:")
print(data[sensor_columns].isnull().sum())

# --------------------------------------------------
# 6. Remove rows with missing sensor values
# --------------------------------------------------

rows_before = len(data)

data = data.dropna(
    subset=sensor_columns
).copy()

print(
    "\nRows removed due to missing values:",
    rows_before - len(data)
)

# --------------------------------------------------
# 7. Remove duplicate samples
# --------------------------------------------------

rows_before = len(data)

if "sample_index" in data.columns:
    data = data.drop_duplicates(
        subset=["source_file", "sample_index"]
    )
else:
    data = data.drop_duplicates()

print(
    "Duplicate rows removed:",
    rows_before - len(data)
)

data = data.reset_index(drop=True)

# --------------------------------------------------
# 8. Download participant information
# --------------------------------------------------

subject_response = requests.get(
    subject_url, timeout=60
)

subject_response.raise_for_status()

subjects = pd.read_csv(
    BytesIO(subject_response.content)
)

subjects.columns = subjects.columns.str.strip()

# Rename participant ID column
if "code" in subjects.columns:
    subjects = subjects.rename(
        columns={"code": "subject_id"}
    )
elif "subject_id" not in subjects.columns:
    raise ValueError(
        "Participant ID column not found."
    )

# Convert participant IDs to numeric integers
data["subject_id"] = pd.to_numeric(
    data["subject_id"], errors="coerce"
).astype("Int64")

subjects["subject_id"] = pd.to_numeric(
    subjects["subject_id"], errors="coerce"
).astype("Int64")

# Remove duplicate participant records
subjects = subjects.drop_duplicates(
    subset=["subject_id"]
)

# Merge participant details into the main dataset
data = data.merge(
    subjects,
    on="subject_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_subject")
)

print(
    "\nShape after merging participant information:",
    data.shape
)

# --------------------------------------------------
# 9. Validate the cleaned dataset
# --------------------------------------------------

print("\n========== CLEANING SUMMARY ==========")

print("Final dataset shape:", data.shape)

print(
    "Number of sensor features:",
    len(sensor_columns)
)

print(
    "Number of activities:",
    data["activity"].nunique()
)

print(
    "Activities:",
    sorted(data["activity"].dropna().unique())
)

print(
    "Number of participants:",
    data["subject_id"].nunique()
)

print("\nActivity distribution:")
print(data["activity"].value_counts())

print("\nMissing values:")
missing_values = data.isnull().sum()

print(
    missing_values[missing_values > 0]
)

print(
    "\nInfinite sensor values:",
    np.isinf(data[sensor_columns].to_numpy()).sum()
)

if "sample_index" in data.columns:
    print(
        "\nDuplicate file/sample-index pairs:",
        data.duplicated(
            subset=["source_file", "sample_index"]
        ).sum()
    )

print("\nSensor statistics:")
display(data[sensor_columns].describe())

# --------------------------------------------------
# 10. Save the cleaned dataset
# --------------------------------------------------

output_file = "motionsense_cleaned.csv"

data.to_csv(output_file, index=False)

print(
    "\nCleaned dataset saved successfully as:",
    output_file
)

print("Final rows:", len(data))
print("Final columns:", len(data.columns))

# Display first five rows
display(data.head())

Dataset downloaded successfully.
Valid files loaded: 360
Files skipped: 4
Combined dataset shape: (1412865, 18)

Missing sensor values before cleaning:
attitude.roll         0
attitude.pitch        0
attitude.yaw          0
gravity.x             0
gravity.y             0
gravity.z             0
rotationRate.x        0
rotationRate.y        0
rotationRate.z        0
userAcceleration.x    0
userAcceleration.y    0
userAcceleration.z    0
dtype: int64

Rows removed due to missing values: 0
Duplicate rows removed: 0

Shape after merging participant information: (1412865, 22)

========== CLEANING SUMMARY ==========
Final dataset shape: (1412865, 22)
Number of sensor features: 12
Number of activities: 6
Activities: ['downstairs', 'jogging', 'sitting', 'standing', 'upstairs', 'walking']
Number of participants: 24

Activity distribution:
activity
walking       344288
sitting       338778
standing      306427
upstairs      157285
jogging       134231
downstairs    131856
Name: count, dtype: int

,attitude.roll,attitude.pitch,attitude.yaw,gravity.x,gravity.y,gravity.z,rotationRate.x,rotationRate.y,rotationRate.z,userAcceleration.x,userAcceleration.y,userAcceleration.z
count,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06,1.412865e+06
mean,-1.760427e-01,-9.866487e-01,-1.492687e-01,4.213359e-02,7.547222e-01,-1.388248e-01,5.321167e-03,1.458119e-02,1.276031e-02,-3.910640e-03,4.424380e-02,3.927177e-02
std,1.566652e+00,4.940860e-01,1.544769e+00,3.388300e-01,3.332649e-01,4.283231e-01,1.292916e+00,1.226470e+00,8.080411e-01,3.282090e-01,5.275663e-01,3.762121e-01
min,-3.141590e+00,-1.569974e+00,-3.141588e+00,-9.998220e-01,-9.999930e-01,-9.999990e-01,-1.736579e+01,-1.841441e+01,-1.215124e+01,-6.369264e+00,-5.673592e+00,-7.743481e+00
25%,-1.480050e+00,-1.358278e+00,-1.277552e+00,-1.725300e-01,5.837930e-01,-3.922410e-01,-2.860300e-01,-1.669860e-01,-9.733300e-02,-4.751300e-02,-6.213600e-02,-2.361300e-02
50%,1.222180e-01,-1.181032e+00,-1.356530e-01,1.991400e-02,9.249990e-01,-9.690200e-02,-3.940000e-04,3.250000e-04,1.770000e-04,8.910000e-04,4.336000e-03,3.433000e-03
75%,8.763600e-01,-6.233930e-01,8.291430e-01,2.721440e-01,9.775030e-01,8.488200e-02,1.163210e-01,2.312710e-01,1.133280e-01,4.428400e-02,5.176600e-02,1.047770e-01
max,3.141579e+00,1.567175e+00,3.141584e+00,9.999980e-01,1.000000e+00,9.813120e-01,1.046806e+01,1.754312e+01,1.143624e+01,7.120793e+00,7.322722e+00,8.125357e+00



Cleaned dataset saved successfully as: motionsense_cleaned.csv
Final rows: 1412865
Final columns: 22


,sample_index,attitude.roll,attitude.pitch,attitude.yaw,gravity.x,gravity.y,gravity.z,rotationRate.x,rotationRate.y,rotationRate.z,...,userAcceleration.z,activity,activity_code,subject_id,trial,source_file,weight,height,age,gender
0,0,-2.116381,-1.077507,-2.261502,-0.404768,0.880780,0.245713,-1.264215,-1.027909,-0.947909,...,-0.407670,downstairs,dws,12,11,A_DeviceMotion_data/dws_11/sub_12.csv,60,167,33,1
1,1,-2.148154,-1.049759,-2.284278,-0.417081,0.867303,0.271686,-1.162024,-0.269118,-0.848823,...,-0.560291,downstairs,dws,12,11,A_DeviceMotion_data/dws_11/sub_12.csv,60,167,33,1
2,2,-2.153824,-1.026749,-2.297008,-0.432082,0.855621,0.284961,-0.665042,0.520170,-0.726722,...,-0.463275,downstairs,dws,12,11,A_DeviceMotion_data/dws_11/sub_12.csv,60,167,33,1
3,3,-2.142509,-1.012749,-2.290595,-0.445311,0.848291,0.286507,-0.079809,0.055322,-0.604534,...,-0.510293,downstairs,dws,12,11,A_DeviceMotion_data/dws_11/sub_12.csv,60,167,33,1
4,4,-2.130486,-1.007262,-2.274149,-0.452661,0.845372,0.283600,0.456097,-0.186877,-0.441315,...,-0.925049,downstairs,dws,12,11,A_DeviceMotion_data/dws_11/sub_12.csv,60,167,33,1


In [10]:
!pwd
!ls -lh

/content/adaptive-sensor-fusion
total 269M
-rw-r--r-- 1 root root 269M Oct 23 15:09 motionsense_cleaned.csv


In [11]:
%%writefile .gitignore
motionsense_cleaned.csv
*.zip
__pycache__/

Writing .gitignore
